# Remote MLflow deployment

So far, we've been running MLflow locally. That's fine for exploration, but in a real team, you'll want a central server where everyone can log their runs. In this notebook, we'll cover:

1. Connecting the MLflow client to a remote tracking server.
2. Logging runs, registering models, and loading them from that remote server.
3. Comparing the trade-offs between local and remote setups.

## 1. Before you start

You'll need to start the remote MLflow stack first. From the project root, run:

```bash
docker-compose up -d
```

This kicks off PostgreSQL (for metadata), MinIO (for model files), and the MLflow server itself.

## 2. Setting the tracking URI

In [1]:
import sys, os

REPO_ROOT = os.path.abspath(os.pardir)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from src.data_preprocessing import load_data, preprocess, split_data

# Use env var for flexibility; default to localhost when running the Docker stack
REMOTE_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(REMOTE_URI)

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

MLflow version: 3.15.2
Tracking URI: http://localhost:5000


## 3. Checking the connection

It's worth double-checking that the server is actually up before you try to log anything. If the server is down, MLflow will throw an error as soon as you try to start a run.

In [2]:
try:
    experiments = mlflow.search_experiments()
    print(f"Connected. Found {len(experiments)} experiment(s).")
except Exception as e:
    print(f"Cannot reach MLflow server: {e}")
    print("\nMake sure the stack is running: docker-compose up -d")

Connected. Found 1 experiment(s).


## 4. Preparing the data

We're using the same preprocessing as before. The only difference now is that all our logging will go to the remote server instead of a local SQLite file.

In [3]:
EXPERIMENT_NAME = "telco-churn-remote"
mlflow.set_experiment(EXPERIMENT_NAME)

df = load_data()
df = df.sample(n=25000, random_state=42).reset_index(drop=True)
X, y, scaler, feature_names = preprocess(df)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

2026/09/02 16:48:04 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-remote' does not exist. Creating a new experiment.


Experiment: telco-churn-remote
Train: (20000, 30), Test: (5000, 30)


## 5. Logging a run to the remote server

Everything works exactly the same way as before - MLflow just sends the data over the network instead of writing to disk locally.

In [4]:
with mlflow.start_run(run_name="remote-logistic-baseline"):
    mlflow.set_tag("tracking", "remote")
    mlflow.set_tag("model_type", "LogisticRegression")

    model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    mlflow.log_params({"C": 1.0, "max_iter": 1000})
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1_score": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
    })
    mlflow.sklearn.log_model(model, name="model", input_example=X_test.iloc[:5])

    run_id = mlflow.active_run().info.run_id
    print(f"Run logged to remote server. Run ID: {run_id}")

Run logged to remote server. Run ID: 3039512577234b518901f6532ac519c6
🏃 View run remote-logistic-baseline at: http://localhost:5000/#/experiments/1/runs/3039512577234b518901f6532ac519c6
🧪 View experiment at: http://localhost:5000/#/experiments/1


## 6. Querying the remote server

When you call `mlflow.search_runs()`, the client fetches the metadata directly from the remote PostgreSQL database.

In [5]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"],
)

cols = ["run_id", "tags.mlflow.runName", "metrics.f1_score", "metrics.roc_auc", "metrics.accuracy"]
runs_df[[c for c in cols if c in runs_df.columns]].head(10)

,run_id,tags.mlflow.runName,metrics.f1_score,metrics.roc_auc,metrics.accuracy
0,3039512577234b518901f6532ac519c6,remote-logistic-baseline,0.717887,0.720623,0.6732


## 7. Using the remote model registry

We can register models on the remote server just like we did locally. The model files will be stored in MinIO, but MLflow handles all the communication for you.

In [6]:
MODEL_NAME = "TelcoChurnModelRemote"

best_run = runs_df.iloc[0]
model_uri = f"runs:/{best_run['run_id']}/model"
result = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

from mlflow import MlflowClient
client = MlflowClient()
client.set_registered_model_alias(name=MODEL_NAME, alias="champion", version=result.version)

print(f"Registered: {MODEL_NAME} v{result.version} with alias 'champion'")

Successfully registered model 'TelcoChurnModelRemote'.
2026/09/02 17:08:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 3039512577234b518901f6532ac519c6 has no artifacts at artifact path 'model', registering model based on models:/m-e33b7616ade9484aa9fffb2c0270fb66 instead
2026/09/02 17:08:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: TelcoChurnModelRemote, version 1
Created version '1' of model 'TelcoChurnModelRemote'.


Registered: TelcoChurnModelRemote v1 with alias 'champion'


## 8. Loading the remote model

When you load a model by its alias, MLflow downloads the artifacts from the remote server. You don't need any local copies of the model files to make this work.

In [ ]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
predictions = loaded_model.predict(X_test)

print(f"Loaded model type: {type(loaded_model)}")
print(f"Sample predictions: {predictions[:5]}")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

2026/09/02 17:11:56 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for conda.yaml. Retries remaining: 7
2026/09/02 17:11:56 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for input_example.json. Retries remaining: 7
2026/09/02 17:11:56 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for MLmodel. Retries remaining: 7
2026/09/02 17:11:58 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for model.skops. Retries remaining: 7
2026/09/02 17:13:39 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for MLmodel. Retries remaining: 6
2026/09/02 17:13:39 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for conda.yaml. Retries remaining: 6
2026/09/02 17:13:39 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for model.skops. Retries remaining: 6
2026/09/02 17:13:40 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 f

KeyboardInterrupt: 

2026/09/02 17:24:07 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for MLmodel. Retries remaining: 1
2026/09/02 17:24:08 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for input_example.json. Retries remaining: 1
2026/09/02 17:24:08 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for model.skops. Retries remaining: 1
2026/09/02 17:26:01 WARNING mlflow.utils.file_utils: Successfully removed /tmp/tmpzcepsckk/conda.yaml
2026/09/02 17:27:36 WARNING mlflow.utils.file_utils: Successfully removed /tmp/tmpzcepsckk/input_example.json
2026/09/02 17:27:36 INFO mlflow.store.artifact.http_artifact_repo: Retrying 1 failed chunk(s) for python_env.yaml. Retries remaining: 7
2026/09/02 17:27:37 WARNING mlflow.utils.file_utils: Successfully removed /tmp/tmpzcepsckk/MLmodel
2026/09/02 17:27:37 WARNING mlflow.utils.file_utils: Successfully removed /tmp/tmpzcepsckk/model.skops


## 9. Local vs. remote: what's the difference?

| Aspect | Local (`sqlite:///`) | Remote (Docker/Cloud) |
|--------|---------------------|----------------------------------|
| **Backend** | SQLite file | PostgreSQL (usually) |
| **Artifacts** | Local `./mlruns/` folder | MinIO, S3, or GCS |
| **Sharing** | Just your machine | The whole team |
| **UI** | Run `mlflow ui` yourself | Hosted on the server |
| **Config** | Hardcoded path | Environment variables |

## 10. Switching environments with environment variables

In a real production pipeline, you'll probably use `MLFLOW_TRACKING_URI` to switch between staging and production environments without changing your code:

```bash
export MLFLOW_TRACKING_URI=http://mlflow-server:5000
python train.py
```